# KVQuant Implementation -- full-precision baseline, GSM8K only

Stripped-down variant of
`KVQuant_Baseline_seed42_WikiText256_OtherDatasets1024.ipynb` that keeps
ONLY what GSM8K needs. WikiText-103, ARC-Challenge, and HellaSwag are
removed entirely -- their loading, scoring, and driver cells, plus the
shared machinery only those three datasets needed (`measure_chunk_kv_memory`,
the multiple-choice scoring block) -- since GSM8K measures its own memory
inline in `generate_gsm8k_kvquant` and never touches either.

Everything GSM8K-relevant is otherwise identical to the source notebook:
real `generate()`-based timing with TBT excluding TTFT, perplexity measured
live on the model's own generated answer (no separate teacher-forced pass),
random sampling of up to 1,024 valid questions with seed 42, per-prompt CSV
export, and download-retry-with-cache-clear robustness. Memory is measured
directly from the real KV cache tensors this model actually produces (no
outlier-hook simulation needed, since nothing here is quantized).

Run cells top to bottom. Needs a GPU runtime.

## Setup

In [1]:
!hostname

DESKTOP-32E0L5J


In [2]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods (kept
# identical to the 2-bit/3-bit/4-bit notebooks even though this one never
# clones/patches the KVCacheCompression repo, so there's no risk of a
# transformers/tokenizers/etc. version mismatch skewing the comparison).

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- Llama-3.1-8B is GATED: this will fail to load without a token that has accepted the Meta license at https://huggingface.co/meta-llama/Llama-3.1-8B")

print("Block 1 finished. Now run Block 2.")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Block 2 - Imports, GPU check

import gc
import math
import os
import re
import shutil
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.bfloat16 if HAS_CUDA else torch.float32


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

# NOTE: clear_hf_dataset_cache/robust_call are defined here (not in Helper
# Functions, below) for consistency with the KVQuant-family notebooks,
# where the Fisher calibration cell needs them available early in Setup.
# This notebook has no such dependency itself (no calibration step), but
# keeping the split identical across the whole notebook family avoids
# Helper Functions containing different things in different notebooks.
# sync_if_cuda/clear_memory have no early dependency anywhere and live in
# Helper Functions with the rest of the genuinely cross-dataset machinery.

In [ ]:
# Block 3 - Experiment settings.
# Sampling policy for GSM8K (the only dataset in this notebook): up to
# 1,024 random valid questions, deterministic seed 42. Sampling happens
# after validity filtering; selected indices are sorted back into source
# order so the subset is random while evaluation order stays stable. No
# ABITS/SPARSITY_THRESHOLD/quantizer settings here -- this notebook never
# quantizes anything.

LOCAL_MODEL_PATH = "/content/llama-3.1-8b"
HF_MODEL_ID = "meta-llama/Llama-3.1-8B"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 42
QA_EVAL_SAMPLES = 2048
GSM8K_MAX_NEW_TOKENS = 256
METHOD_NAME = "kvquant_baseline_full_precision"

random.seed(SHARED_SEED)
np.random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SHARED_SEED)

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation step by step, then end with exactly this format:\n"
    "#### <final number>\n\n"

    "Question: There are 15 trees in the grove. Grove workers will plant trees today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n"
    "Answer: There are 15 trees originally. After planting, there are 21 trees. So the workers planted 21 - 15 = 6 trees.\n"
    "#### 6\n\n"

    "Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n"
    "Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5 cars.\n"
    "#### 5\n\n"

    "Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n"
    "Answer: Leah and her sister started with 32 + 42 = 74 chocolates. After eating 35, they have 74 - 35 = 39 left.\n"
    "#### 39\n\n"

    "Question: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\n"
    "Answer: Jason started with 20 lollipops and now has 12. So he gave away 20 - 12 = 8 lollipops.\n"
    "#### 8\n\n"

    "Question: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\n"
    "Answer: Shawn started with 5 toys. He got 2 from mom and 2 from dad, which is 2 + 2 = 4 more toys. 5 + 4 = 9 toys total.\n"
    "#### 9\n\n"

    "Question: There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?\n"
    "Answer: 4 days from Monday to Thursday, with 5 computers installed each day, is 4 * 5 = 20 computers added. 9 + 20 = 29 computers total.\n"
    "#### 29\n\n"

    "Question: Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?\n"
    "Answer: Michael started with 58 golf balls. After losing 23 on Tuesday, he had 58 - 23 = 35. After losing 2 more on Wednesday, he had 35 - 2 = 33 golf balls.\n"
    "#### 33\n\n"

    "Question: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\n"
    "Answer: Five bagels at $3 each cost 5 * 3 = 15 dollars. Olivia started with $23, so she has 23 - 15 = 8 dollars left.\n"
    "#### 8\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("Random sampling seed:", SHARED_SEED)
print("GSM8K random example target:", QA_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)


In [ ]:
# Block - Load tokenizer + the untouched full-precision model. No repo
# clone, no patches, no Fisher calibration, no Quantize step -- none of
# that machinery is needed for a model that's never quantized. This is the
# exact same loading code the combined v3 notebook used for model_fp,
# unchanged, so the baseline is byte-for-byte comparable across notebooks.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "sdpa",
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print("Loading full-precision baseline model...")
model_fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_fp = model_fp.to(DEVICE)
model_fp.eval()
model_fp.config.use_cache = True

device = next(model_fp.parameters()).device
print("model_fp: full-precision baseline, loaded and ready.")

## Helper Functions

Shared inference machinery used across all three datasets (WikiText-103,
GSM8K, ARC-Challenge).

In [ ]:
# Block - sync_if_cuda/clear_memory: used across every timed inference
# loop in this notebook (WikiText-103, GSM8K, ARC-Challenge) for
# timing-safe GPU synchronization and between-dataset memory cleanup.


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

In [ ]:
def seeded_subset(items, max_samples, seed=SHARED_SEED):
    """Select a reproducible random subset, then restore source order.

    A fresh RNG is created on every call, so running notebook sections in a
    different order cannot change which examples are selected.
    """
    items = list(items)

    sample_count = min(int(max_samples), len(items))

    selected_indices = sorted(
        random.Random(int(seed)).sample(range(len(items)), sample_count)
    )

    return [items[index] for index in selected_indices], selected_indices

In [ ]:
def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err

## GSM8K

In [ ]:
# Block - GSM8K loading: combine all splits, then random sampling.
# Load train + test splits, build the complete list of valid question/answer
# pairs, then select up to 1,024 examples with seed 42. This creates a
# reproducible random sample across the entire GSM8K dataset.

def extract_gsm8k_gold_answer(answer_text):
    match = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not match:
        return None
    try:
        return float(match.group(1).replace(",", ""))
    except ValueError:
        return None

gsm8k_train = robust_call(
    load_dataset,
    "gsm8k",
    "main",
    split="train",
    desc="GSM8K train load",
    on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

gsm8k_test = robust_call(
    load_dataset,
    "gsm8k",
    "main",
    split="test",
    desc="GSM8K test load",
    on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

# Combine all official GSM8K splits
gsm8k_all = list(gsm8k_train) + list(gsm8k_test)
all_gsm8k_pairs = []

for item in gsm8k_all:
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        all_gsm8k_pairs.append({
            "question": item["question"],
            "gold": gold,
            "gold_text": item["answer"],
        })

gsm8k_qa_pairs, gsm8k_selected_indices = seeded_subset(
    all_gsm8k_pairs,
    QA_EVAL_SAMPLES,
    SHARED_SEED,
)

print(
    f"GSM8K: {len(all_gsm8k_pairs)} valid questions available; "
    f"selected {len(gsm8k_qa_pairs)} random questions "
    f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
)

print(
    "GSM8K selected valid-item indices (first 20):",
    gsm8k_selected_indices[:20]
)

In [ ]:
# Block - GSM8K generation with the full-precision baseline. Uses the real
# model.generate() call -- prefill processes the whole prompt in one shot,
# then decode continues one token at a time exactly like a normal
# generate() loop. A StoppingCriteria timestamps every generated token so
# TTFT/TBT come from this SAME call that produces the graded answer -- not
# a separate probe. Memory is measured in a SEPARATE, untimed pass after
# generate() returns, so it cannot affect any timing number.
#
# Perplexity is measured on the model's OWN generated answer, live from
# this same call -- no separate teacher-forced pass. generate() is called
# with output_scores=True, return_dict_in_generate=True, which returns the
# per-step logits it already computes internally (no extra forward pass)
# alongside the token ids. After generate() returns (i.e. fully outside
# the timed window), we scan the generated tokens for four consecutive
# '#' characters (the "####" marker GSM8K answers use), then accumulate
# the log-probability of each token the model chose for itself, starting
# right after the marker and stopping the moment another '#' appears. If
# the model never generates "####" at all, perplexity is None for that
# question.


def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums:
            return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


class _TimingCriteria(StoppingCriteria):
    def __init__(self):
        self.token_times = []

    def __call__(self, input_ids, scores, **kwargs):
        self.token_times.append(time.perf_counter())
        return False


@torch.no_grad()
def generate_gsm8k(model_obj, question, bits_per_element, tracker=None):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    timing = _TimingCriteria()
    sync_if_cuda()
    gen_start = time.perf_counter()
    gen_out = model_obj.generate(
        **enc, max_new_tokens=GSM8K_MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        stopping_criteria=StoppingCriteriaList([timing]),
        output_scores=True, return_dict_in_generate=True,
    )
    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    gen_ids = gen_out.sequences
    step_scores = gen_out.scores

    gen_text = tokenizer.decode(gen_ids[0][prompt_len:], skip_special_tokens=True)
    gen_text = gen_text.split("Question:")[0]
    gen_token_count = len(tokenizer(gen_text, add_special_tokens=False)["input_ids"])

    n_generated = len(timing.token_times)
    if n_generated > 0:
        ttft_sec = timing.token_times[0] - gen_start
    else:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated

    # Perplexity of the model's own predicted final number -- everything
    # below runs after gen_end, so none of it can affect any timing number.
    # step_scores[i] are the logits that chose the i-th generated token
    # (already computed by generate() above -- no extra forward pass).
    answer_token_ids = gen_ids[0][prompt_len: prompt_len + len(step_scores)].tolist()
    hash_streak = 0
    in_answer_span = False
    nll_sum = 0.0
    scored = 0
    for i, tok_id in enumerate(answer_token_ids):
        tok_text = tokenizer.decode([tok_id])
        if in_answer_span:
            if "#" in tok_text:
                break
            log_probs = torch.log_softmax(step_scores[i][0].float(), dim=-1)
            nll_sum += -log_probs[tok_id].item()
            scored += 1
        else:
            for ch in tok_text:
                hash_streak = hash_streak + 1 if ch == "#" else 0
            if hash_streak >= 4:
                in_answer_span = True
    perplexity = math.exp(min(nll_sum / scored, 50.0)) if scored > 0 else None

    # Untimed pass over the full generated sequence, purely to measure
    # memory -- real cache tensor bytes for the full-precision baseline
    # (no outlier-hook tracker needed, nothing here is quantized). Does
    # not affect the timed generate() call above.
    if tracker is not None:
        reset_outlier_tracker(tracker)
        model_obj(input_ids=gen_ids, use_cache=False, return_dict=True)
        peak_bytes = measure_bytes_from_tracker(tracker, bits_per_element)
    else:
        cache_outputs = model_obj(input_ids=gen_ids, use_cache=True, return_dict=True)
        pkv = cache_outputs.past_key_values
        legacy = pkv.to_legacy_cache() if hasattr(pkv, "to_legacy_cache") else pkv
        peak_bytes = sum(t.numel() * t.element_size() for layer_kv in legacy for t in layer_kv)

    return {
        "prompt": prompt,
        "gen_text": gen_text, "gen_token_count": gen_token_count,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": total_tokens,
        "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


In [ ]:
# Block - GSM8K driver: run every question through generate_gsm8k_kvquant
# (accuracy + TTFT/TBT/latency + memory + perplexity, all from the SAME
# real generation call -- perplexity is measured live on the model's own
# generated answer, no separate teacher-forced pass) -- against the
# full-precision baseline.


def evaluate_gsm8k(qa_pairs, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 100
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        result = generate_gsm8k(model_obj, qa["question"], bits_per_element, tracker)
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

        ppl = result["perplexity"]
        if ppl is not None:
            ppl_values.append(ppl)

        per_question_records.append({
            "question_index": q_idx,
            "prompt": result["prompt"],
            "generated_response": result["gen_text"],
            "generated_response_tokens": result["gen_token_count"],
            "gold_answer": qa["gold"],
            "predicted_answer": pred,
            "correct": int(is_correct),
            "perplexity": ppl,
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_gsm8k_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question GSM8K rows to {_per_prompt_path}")

    return {
        "dataset": "GSM8K",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


gsm8k_results = [
    evaluate_gsm8k(gsm8k_qa_pairs, model_fp, 16, METHOD_NAME),
]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)

## Save Results

In [ ]:
# Block - Save GSM8K results to CSV. This notebook only ever produces a
# single "kvquant_baseline_full_precision" GSM8K row -- WikiText-103,
# ARC-Challenge, and HellaSwag are handled by the full baseline notebook,
# not here. Saved under a distinct "_gsm8k_only_" filename so it can
# never collide with (or get overwritten by) the full notebook's own
# 4-dataset summary CSV in the same Drive folder. The per-question CSV
# from evaluate_gsm8k_kvquant still uses the shared METHOD_NAME-based
# filename, since it's the exact same GSM8K data either way.

results_df = gsm8k_results_df[[
    "dataset", "method", "perplexity", "accuracy",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_mb", "average_memory_mb",
]]
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)

_path = "/content/drive/MyDrive/KVQuant_v3_Results/kvquant_baseline_full_precision_gsm8k_only_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")

In [ ]:
from google.colab import runtime
runtime.unassign()